# Delta Lake com Apache Spark

Este notebook demonstra como usar Delta Lake com PySpark para criar uma tabela transacional local.

- Criação e escrita de tabela Delta
- Leitura e consulta de dados
- Atualização de registros
- Time travel

In [2]:
import os
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip


builder = (
    SparkSession.builder
    .appName("DeltaLakeStudy")
    .master("local[*]")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()

delta_path = os.path.abspath("tmp/delta_table")
os.makedirs(os.path.dirname(delta_path), exist_ok=True)

data = [(1, "Alice", 1200), (2, "Bob", 1500), (3, "Carol", 1600)]
df = spark.createDataFrame(data, ["id", "name", "salary"])

df.write.format("delta").mode("overwrite").save(delta_path)
print(f"Tabela Delta escrita em: {delta_path}")

26/04/26 19:15:16 WARN Utils: Your hostname, DESKTOP-2SHE2K2 resolves to a loopback address: 127.0.1.1; using 172.20.252.12 instead (on interface eth0)
26/04/26 19:15:16 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Ivy Default Cache set to: /home/gabriel/.ivy2/cache
The jars for the packages stored in: /home/gabriel/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-977f3ebb-78aa-4515-862e-1ab025957dbc;1.0
	confs: [default]


:: loading settings :: url = jar:file:/home/gabriel/.cache/pypoetry/virtualenvs/datalakehouse-study-AClZzX3p-py3.12/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 166ms :: artifacts dl 8ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.2.0 from central in [default]
	io.delta#delta-storage;3.2.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0   |   0   |   0   ||   3   |   0   |
	---------------------------------------------------------------------
:: retrieving :: org.apache.spark#spark-submit-parent-977f3ebb-78aa-4515-862e-1ab025957dbc
	confs: [default]
	0 artifacts copied, 3 already retrieved (0kB/6ms)
26/04/26 19:15:17

Tabela Delta escrita em: /home/gabriel/GitHub/datalakehouse-study/notebooks/tmp/delta_table


In [ ]:
spark.read.format("delta").load(delta_path).show()

spark.sql("DROP TABLE IF EXISTS delta_demo")
spark.sql(f"CREATE TABLE delta_demo USING DELTA LOCATION '{delta_path}'")
spark.sql("SELECT * FROM delta_demo").show()

In [4]:
from delta.tables import DeltaTable

delta_table = DeltaTable.forPath(spark, delta_path)
delta_table.update("id = 1", {"salary": "salary + 400"})
print("Registro atualizado para id = 1")

spark.read.format("delta").load(delta_path).show()

Registro atualizado para id = 1
+---+-----+------+
| id| name|salary|
+---+-----+------+
|  1|Alice|  7000|
|  3|Carol|  1600|
|  2|  Bob|  1500|
+---+-----+------+



In [ ]:
print("Versão 0 - antes da atualização")
spark.read.format("delta").option("versionAsOf", 0).load(delta_path).show()